In [ ]:
#!/usr/bin/env python3
"""
TOPAZ FakeFez Benchmark - Noisy 8-Qubit Path Validation
======================================================
Focuses strictly on:
1. A single seed (Seed 0).
2. The noisiest 8-qubit path on the IBM FakeFez 156-qubit device.
3. Benchmarks:
   - Baseline (Unoptimized, No ZNE)
   - ZNE Alone (Unoptimized, with ZNE)
   - TOPAZ Alone (Optimized, No ZNE)
   - TOPAZ + ZNE (Optimized, with ZNE)
"""

import numpy as np
import scipy.linalg as la
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit_aer.noise import NoiseModel
from qiskit.primitives import EstimatorV2
from qiskit.circuit.library import UnitaryGate
import networkx as nx
import itertools
import warnings
import time

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
N_QUBITS       = 8
DIM            = 2 ** N_QUBITS
MAX_ITERATIONS = 40
REG_LAMBDA     = 1e-3
RUN_SEED       = 0

_I = np.eye(2, dtype=complex)
_X = np.array([[0,1],[1,0]], dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)
_Z = np.array([[1,0],[0,-1]], dtype=complex)

_PAIRS_2Q = {
    'XX': np.kron(_X,_X), 'XY': np.kron(_X,_Y), 'XZ': np.kron(_X,_Z),
    'YX': np.kron(_Y,_X), 'YY': np.kron(_Y,_Y), 'YZ': np.kron(_Y,_Z),
    'ZX': np.kron(_Z,_X), 'ZY': np.kron(_Z,_Y), 'ZZ': np.kron(_Z,_Z),
}

_NN_PAULI_NAMES = ['XX','YY','ZZ']
N_PAIRS  = N_QUBITS - 1
N_TERMS  = N_PAIRS * len(_NN_PAULI_NAMES)   # 21 terms
N_PARAMS = 2 * N_TERMS                       # 42 params

_TERM_INFO = []
for _pi in range(N_PAIRS):
    for _pn in _NN_PAULI_NAMES:
        _TERM_INFO.append({
            'P2q'   : _PAIRS_2Q[_pn],
            'numpy_i': _pi,
            'numpy_j': _pi + 1,
            'qk_qa' : N_QUBITS - 2 - _pi,
            'qk_qb' : N_QUBITS - 1 - _pi,
        })

_PERM = [int(format(i, f'0{N_QUBITS}b')[::-1], 2) for i in range(DIM)]

# ── Setup FakeFez backend ─────────────────────────────────────────────────────
print("Initializing FakeFez (156-qubit IBM device noise emulator)...")
_fake_backend = FakeFez()
_noise_model  = NoiseModel.from_backend(_fake_backend)
_sim          = AerSimulator(noise_model=_noise_model)
print("Ready: FakeFez noise simulator successfully configured.")

# ── DFS to Find the Noisiest 8-Qubit Path ─────────────────────────────────────
print("\nScanning coupling map to locate the Noisiest 8-qubit path...")
props = _fake_backend.properties()
cmap = _fake_backend.configuration().coupling_map
basis_gates = _fake_backend.configuration().basis_gates
two_q_gate = [g for g in basis_gates if g in ['cx', 'cz', 'ecr']][0]

G = nx.Graph()
for edge in cmap:
    q1, q2 = edge
    try: err1 = props.gate_error(two_q_gate, [q1, q2]) or 0.0
    except: err1 = 0.0
    try: err2 = props.gate_error(two_q_gate, [q2, q1]) or 0.0
    except: err2 = 0.0
    err = (err1 + err2) / 2.0
    try: ro1 = props.readout_error(q1) or 0.0
    except: ro1 = 0.0
    try: ro2 = props.readout_error(q2) or 0.0
    except: ro2 = 0.0
    ro = (ro1 + ro2) / 2.0
    G.add_edge(q1, q2, weight=err + 0.1 * ro)

paths = []
def dfs(node, path):
    if len(path) == 8:
        paths.append(path)
        return
    for neighbor in G.neighbors(node):
        if neighbor not in path:
            dfs(neighbor, path + [neighbor])

for node in G.nodes:
    dfs(node, [node])

path_scores = []
for p in paths:
    score = sum(G[p[i]][p[i+1]]['weight'] for i in range(7))
    path_scores.append((score, p))

path_scores.sort(key=lambda x: x[0])
noisiest_8 = path_scores[-1][1]
print(f"Target Layout (Noisiest 8): {noisiest_8} (Cumulative Score: {path_scores[-1][0]:.4f})")

# ── Core Helpers ──────────────────────────────────────────────────────────────
def _qiskit_dm_to_numpy(rho_q):
    return rho_q[np.ix_(_PERM, _PERM)]

def _apply_gate(U_2q, psi_tensor, qi, qj):
    N = psi_tensor.ndim
    axes = [qi, qj] + [k for k in range(N) if k != qi and k != qj]
    psi  = np.transpose(psi_tensor, axes).reshape(4, -1)
    psi  = (U_2q @ psi).reshape((2, 2) + (2,)*(N-2))
    return np.transpose(psi, np.argsort(axes))

def compute_ideal_state(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    psi   = np.zeros((2,)*N_QUBITS, dtype=complex)
    psi[(0,)*N_QUBITS] = 1.0
    for j, t in enumerate(_TERM_INFO):
        psi = _apply_gate(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), psi, t['numpy_i'], t['numpy_j'])
    return psi.flatten()

def _upte_reg_penalty(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    total = 0.0
    for _pi in range(N_PAIRS):
        B = np.zeros((4,4), dtype=complex)
        for _pp, _pn in enumerate(_NN_PAULI_NAMES):
            j = _pi * len(_NN_PAULI_NAMES) + _pp
            B += rho_n[j] * la.expm(-1j * rho_n[j] * tau[j] * _PAIRS_2Q[_pn])
        dev = B.conj().T @ B - np.eye(4, dtype=complex)
        total += np.real(np.trace(dev.conj().T @ dev))
    return total

def apply_fake_backend_noise(params, layout):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), [t['qk_qa'], t['qk_qb']])
    
    qc_t = transpile(qc, _fake_backend, initial_layout=layout, optimization_level=1)
    qc_t.save_density_matrix(layout)
    
    result = _sim.run(qc_t, shots=1).result()
    return _qiskit_dm_to_numpy(np.array(result.data(0)['density_matrix']))

def noise_aware_objective(params, psi_target, layout):
    psi_ideal = compute_ideal_state(params)
    rho_noisy = apply_fake_backend_noise(params, layout)
    ideal_fid = float(np.abs(np.vdot(psi_target, psi_ideal))**2)
    noisy_fid = float(np.clip(np.real(psi_target.conj() @ rho_noisy @ psi_target), 0, 1))
    loss      = 1.0 - noisy_fid + REG_LAMBDA * _upte_reg_penalty(params)
    return loss, noisy_fid, ideal_fid, rho_noisy

# ── MMA Optimizer ──────────────────────────────────────────────────────────────
class MMAOptimizer:
    def __init__(self, n, move_limit=0.4, gamma=0.5):
        self.n, self.move_limit, self.gamma = n, move_limit, gamma
        self.L = self.U = self.prev_params = self.prev_loss = None
    def init(self, p0, delta=0.6):
        self.L, self.U = p0 - delta, p0 + delta
        self.prev_params = p0.copy()
    def step(self, x, grad):
        x_new = np.zeros_like(x)
        for i in range(self.n):
            g, xi, Li, Ui = grad[i], x[i], self.L[i], self.U[i]
            pi = abs(g)*(Ui-xi)**2 if g < 0 else 0.0
            qi = abs(g)*(xi-Li)**2 if g >= 0 else 0.0
            dn = pi/(Ui-xi+1e-12)**2 + qi/(xi-Li+1e-12)**2
            if dn > 1e-12:
                x_new[i] = xi + (pi/(Ui-xi+1e-12) - qi/(xi-Li+1e-12)) / dn
            else:
                x_new[i] = xi
            x_new[i] = np.clip(x_new[i], max(xi-self.move_limit, Li+1e-6), min(xi+self.move_limit, Ui-1e-6))
        return x_new
    def update(self, x, loss):
        good = self.prev_loss is None or loss < self.prev_loss - 1e-8
        s = 1.2/self.gamma if good else self.gamma
        self.L = x - s*(x - self.L);  self.U = x + s*(self.U - x)
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.prev_params = x.copy();  self.prev_loss = loss

_prev_grad = None

def hybrid_gradient(params, psi_target, layout):
    global _prev_grad
    grad = np.zeros_like(params)
    for i in range(N_TERMS):
        pp = np.clip(params.copy(), 1e-6, None); pp[i] += 1e-4
        pm = np.clip(params.copy(), 1e-6, None); pm[i] -= 1e-4
        grad[i] = (noise_aware_objective(pp, psi_target, layout)[0] - noise_aware_objective(pm, psi_target, layout)[0]) / 2e-4
    for i in range(N_TERMS, N_PARAMS):
        pp = params.copy(); pp[i] += np.pi/4
        pm = params.copy(); pm[i] -= np.pi/4
        grad[i] = (noise_aware_objective(pp, psi_target, layout)[0] - noise_aware_objective(pm, psi_target, layout)[0]) / (np.pi/2)
    if _prev_grad is None:
        _prev_grad = grad.copy(); return grad
    g = 0.4*grad + 0.6*_prev_grad
    _prev_grad = g.copy(); return g

def run_dual_mma(init_params, psi_target, layout, max_iter=MAX_ITERATIONS):
    global _prev_grad
    _prev_grad = None
    mma_r = MMAOptimizer(N_TERMS, move_limit=0.2); mma_r.init(init_params[:N_TERMS], delta=0.4)
    mma_t = MMAOptimizer(N_TERMS, move_limit=0.6); mma_t.init(init_params[N_TERMS:], delta=0.8)
    cur = init_params.copy()
    cur_loss, cur_fid, cur_ifid, _ = noise_aware_objective(cur, psi_target, layout)
    print(f'  Optimization Start: NoisyFid={cur_fid:.4f}  IdealFid={cur_ifid:.4f}')
    best_fid, best_params, stag = cur_fid, cur.copy(), 0
    for it in range(max_iter):
        grad  = hybrid_gradient(cur, psi_target, layout)
        new   = np.concatenate([mma_r.step(cur[:N_TERMS], grad[:N_TERMS]),
                                 mma_t.step(cur[N_TERMS:], grad[N_TERMS:])])
        new_loss, new_fid, _, _ = noise_aware_objective(new, psi_target, layout)
        delta = new_fid - cur_fid
        if new_loss < cur_loss - 1e-6 or delta > -1e-5:
            cur, cur_loss, cur_fid = new, new_loss, new_fid
            if cur_fid > best_fid: best_fid, best_params, stag = cur_fid, cur.copy(), 0
            else: stag += 1
            mma_r.move_limit = min(0.4, mma_r.move_limit*(1.4 if delta>0.01 else 1.2 if delta>0.001 else 0.9))
        else:
            mma_r.move_limit = max(0.02, mma_r.move_limit*0.8); stag += 1
        mma_t.move_limit = mma_r.move_limit * 2.0
        mma_r.update(cur[:N_TERMS], cur_loss)
        mma_t.update(cur[N_TERMS:], cur_loss)
        if (it + 1) % 5 == 0 or it == max_iter - 1:
            print(f'    Iter {it+1:2d}: NoisyFid={cur_fid:.4f} (Δ={delta:+.4f})')
        if stag > 15: 
            print(f'    Optimisation Stagnated at iter {it+1}.')
            break
    print(f'  Optimisation Complete. Best NoisyFid: {best_fid:.4f}')
    return best_params

def build_target(seed=None):
    rng = np.random.default_rng(seed)
    terms = []
    for i in range(N_QUBITS-1):
        for p in ['XX','YY','ZZ']:
            s = ['I']*N_QUBITS; s[i]=p[0]; s[i+1]=p[1]; terms.append(''.join(s))
    return la.expm(-1j * SparsePauliOp(terms, coeffs=rng.uniform(-0.5,0.5,len(terms))).to_matrix(sparse=False))

# ── ZNE Benchmarking ──────────────────────────────────────────────────────────
def build_circuit(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), [t['qk_qa'], t['qk_qb']])
    return qc

paulis = []
coeffs = []
for bits in itertools.product([0, 1], repeat=N_QUBITS):
    p = ''.join(['Z' if b else 'I' for b in bits])
    paulis.append(p)
    coeffs.append(1.0 / (2**N_QUBITS))
projector_obs = SparsePauliOp(paulis, coeffs=coeffs)

def eval_fidelity(qc_ansatz, layout, use_zne=False):
    qc_eval = qc_ansatz.copy()
    qc_eval.append(UnitaryGate(U_target.conj().T), range(N_QUBITS))
    qc_t = transpile(qc_eval, _fake_backend, initial_layout=layout, optimization_level=1)
    
    res_level = 2 if use_zne else 0
    estimator = EstimatorV2(_fake_backend, options={'resilience_level': res_level})
    
    try:
        job = estimator.run([(qc_t, projector_obs)])
        result = job.result()[0]
        return float(result.data.evs)
    except Exception as e:
        print(f"Estimator error: {e}")
        return 0.0

# ── Main Sweep Execution ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*70)
    print(f"RUNNING SINGLE SEED ({RUN_SEED}) BENCHMARK ON NOISIEST 8 LAYOUT")
    print("="*70)

    # 1. Build Target State
    U_target   = build_target(seed=0)
    psi_0      = np.zeros(DIM, dtype=complex); psi_0[0] = 1.0
    psi_target = U_target @ psi_0

    # 2. Initialize Parameters using defined RUN_SEED
    rng = np.random.default_rng(RUN_SEED + 42)
    init_params = np.concatenate([
        rng.uniform(0.1, 0.4, N_TERMS),
        rng.uniform(0.1, np.pi, N_TERMS),
    ])

    qc_baseline = build_circuit(init_params)

    # 3. Optimize with TOPAZ (MMA)
    print(f"\n[Step 1/2] Running TOPAZ Structural MMA Optimization (Seed {RUN_SEED})...")
    t0 = time.time()
    best_params = run_dual_mma(init_params, psi_target, noisiest_8, MAX_ITERATIONS)
    elapsed = time.time() - t0
    print(f"TOPAZ MMA Optimization completed in {elapsed:.1f}s.")

    qc_topaz = build_circuit(best_params)

    # 4. Comparative Evaluation
    print("\n[Step 2/2] Running Comparative Fidelity Evaluations via FakeFez Backend...")
    
    print("\nCalculating: 1. Raw Baseline (Unoptimized, No ZNE)...")
    fid_baseline = eval_fidelity(qc_baseline, noisiest_8, use_zne=False)
    
    print("Calculating: 2. ZNE Alone (Unoptimized + ZNE)...")
    fid_zne_alone = eval_fidelity(qc_baseline, noisiest_8, use_zne=True)
    
    print("Calculating: 3. TOPAZ Alone (Optimized, No ZNE)...")
    fid_topaz_alone = eval_fidelity(qc_topaz, noisiest_8, use_zne=False)
    
    print("Calculating: 4. TOPAZ + ZNE (Optimized + ZNE)...")
    fid_topaz_zne = eval_fidelity(qc_topaz, noisiest_8, use_zne=True)

    # 5. Print Results Summary
    print("\n" + "="*70)
    print(" FAKEFEZ NOISY-8 BENCHMARK RESULTS")
    print("="*70)
    print(f"  Target Layout (Noisiest 8):         {noisiest_8}")
    print(f"  Random Seed Config:                 Seed {RUN_SEED}")
    print("-" * 70)
    print(f"  [A] Baseline (Unmitigated)         : {fid_baseline:.4f}")
    print(f"  [B] ZNE Alone                      : {fid_zne_alone:.4f}  (Improvement vs Baseline: {fid_zne_alone - fid_baseline:+.4f})")
    print(f"  [C] TOPAZ Alone                    : {fid_topaz_alone:.4f}  (Improvement vs Baseline: {fid_topaz_alone - fid_baseline:+.4f})")
    print(f"  [D] TOPAZ + ZNE                    : {fid_topaz_zne:.4f}  (Improvement vs Baseline: {fid_topaz_zne - fid_baseline:+.4f})")
    print("-" * 70)
    print(f"  Synergy (TOPAZ+ZNE vs ZNE Alone)   : {fid_topaz_zne - fid_zne_alone:+.4f}")
    print(f"  Synergy (TOPAZ+ZNE vs TOPAZ Alone) : {fid_topaz_zne - fid_topaz_alone:+.4f}")
    print("="*70 + "\n")


/opt/miniconda3/envs/QISKIT/lib/python3.14/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Initializing FakeFez (156-qubit IBM device noise emulator)...
Ready: FakeFez noise simulator successfully configured.

Scanning coupling map to locate the Noisiest 8-qubit path...
Target Layout (Noisiest 8): [33, 32, 31, 30, 29, 28, 27, 17] (Cumulative Score: 3.0413)

RUNNING SINGLE SEED (0) BENCHMARK ON NOISIEST 8 LAYOUT

[Step 1/2] Running TOPAZ Structural MMA Optimization (Seed 0)...
  Optimization Start: NoisyFid=0.0027  IdealFid=0.0894
    Iter  5: NoisyFid=0.0201 (Δ=-0.0201)
    Iter 10: NoisyFid=0.0528 (Δ=+0.0328)
    Iter 15: NoisyFid=0.0528 (Δ=-0.0170)
    Iter 20: NoisyFid=0.0528 (Δ=-0.0000)
    Iter 25: NoisyFid=0.0528 (Δ=-0.0527)
    Optimisation Stagnated at iter 26.
  Optimisation Complete. Best NoisyFid: 0.0528
TOPAZ MMA Optimization completed in 3031.3s.

[Step 2/2] Running Comparative Fidelity Evaluations via FakeFez Backend...

Calculating: 1. Raw Baseline (Unoptimized, No ZNE)...


TypeError: EstimatorV2.__init__() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given

In [2]:
#!/usr/bin/env python3
"""
Separate script to evaluate only ZNE and ZNE+TOPAZ (TOPAZ optimisation + Zero‑Noise Extrapolation).
The code mirrors the original TOPAZ_FAKEFEZ_zneTOPAZ.ipynb notebook but skips the baseline
and TOPAZ‑only fidelity runs, saving execution time when you only want to verify ZNE.
"""

import time
import numpy as np
import scipy.linalg as la
import itertools
import networkx as nx
import warnings
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit.primitives import EstimatorV2
from qiskit.circuit.library import UnitaryGate

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
N_QUBITS = 8
DIM = 2 ** N_QUBITS
MAX_ITERATIONS = 40
REG_LAMBDA = 1e-3
RUN_SEED = 0

# ── Helper matrices ────────────────────────────────────────────────────────
_I = np.eye(2, dtype=complex)
_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_Z = np.array([[1, 0], [0, -1]], dtype=complex)

_PAIRS_2Q = {
    'XX': np.kron(_X, _X), 'XY': np.kron(_X, _Y), 'XZ': np.kron(_X, _Z),
    'YX': np.kron(_Y, _X), 'YY': np.kron(_Y, _Y), 'YZ': np.kron(_Y, _Z),
    'ZX': np.kron(_Z, _X), 'ZY': np.kron(_Z, _Y), 'ZZ': np.kron(_Z, _Z),
}
_NN_PAULI_NAMES = ['XX', 'YY', 'ZZ']
N_PAIRS = N_QUBITS - 1
N_TERMS = N_PAIRS * len(_NN_PAULI_NAMES)   # 21
N_PARAMS = 2 * N_TERMS                       # 42

_TERM_INFO = []
for _pi in range(N_PAIRS):
    for _pn in _NN_PAULI_NAMES:
        _TERM_INFO.append({
            'P2q': _PAIRS_2Q[_pn],
            'numpy_i': _pi,
            'numpy_j': _pi + 1,
            'qk_qa': N_QUBITS - 2 - _pi,
            'qk_qb': N_QUBITS - 1 - _pi,
        })

_PERM = [int(format(i, f'0{N_QUBITS}b')[::-1], 2) for i in range(DIM)]

# ── FakeFez backend setup ──────────────────────────────────────────────────
print("Initializing FakeFez (156‑qubit IBM device noise emulator)...")
_fake_backend = FakeFez()
_noise_model = NoiseModel.from_backend(_fake_backend)
_sim = AerSimulator(noise_model=_noise_model)
print("Ready: FakeFez noise simulator successfully configured.")

# ── Find the noisiest 8‑qubit path ────────────────────────────────────────
print("\nScanning coupling map to locate the Noisiest 8‑qubit path...")
props = _fake_backend.properties()
cmap = _fake_backend.configuration().coupling_map
basis_gates = _fake_backend.configuration().basis_gates
# pick a two‑qubit gate present in the backend
_two_q_gate = [g for g in basis_gates if g in ["cx", "cz", "ecr"]][0]
G = nx.Graph()
for q1, q2 in cmap:
    try:
        err1 = props.gate_error(_two_q_gate, [q1, q2]) or 0.0
    except Exception:
        err1 = 0.0
    try:
        err2 = props.gate_error(_two_q_gate, [q2, q1]) or 0.0
    except Exception:
        err2 = 0.0
    err = (err1 + err2) / 2.0
    try:
        ro1 = props.readout_error(q1) or 0.0
    except Exception:
        ro1 = 0.0
    try:
        ro2 = props.readout_error(q2) or 0.0
    except Exception:
        ro2 = 0.0
    ro = (ro1 + ro2) / 2.0
    G.add_edge(q1, q2, weight=err + 0.1 * ro)

paths = []

def dfs(node, path):
    if len(path) == 8:
        paths.append(path)
        return
    for neighbor in G.neighbors(node):
        if neighbor not in path:
            dfs(neighbor, path + [neighbor])

for node in G.nodes:
    dfs(node, [node])

path_scores = [(sum(G[p[i]][p[i + 1]]['weight'] for i in range(7)), p] for p in paths]
path_scores.sort(key=lambda x: x[0])
noisiest_8 = path_scores[-1][1]
print(f"Target Layout (Noisiest 8): {noisiest_8} (Cumulative Score: {path_scores[-1][0]:.4f})")

# ── Core helper functions ────────────────────────────────────────────────────

def _qiskit_dm_to_numpy(rho_q):
    return rho_q[np.ix_(_PERM, _PERM)]

def _apply_gate(U_2q, psi_tensor, qi, qj):
    N = psi_tensor.ndim
    axes = [qi, qj] + [k for k in range(N) if k != qi and k != qj]
    psi = np.transpose(psi_tensor, axes).reshape(4, -1)
    psi = (U_2q @ psi).reshape((2, 2) + (2,) * (N - 2))
    return np.transpose(psi, np.argsort(axes))

def compute_ideal_state(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]
    psi = np.zeros((2,) * N_QUBITS, dtype=complex)
    psi[(0,) * N_QUBITS] = 1.0
    for j, t in enumerate(_TERM_INFO):
        psi = _apply_gate(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), psi, t['numpy_i'], t['numpy_j'])
    return psi.flatten()

def _upte_reg_penalty(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]
    total = 0.0
    for _pi in range(N_PAIRS):
        B = np.zeros((4, 4), dtype=complex)
        for _pp, _pn in enumerate(_NN_PAULI_NAMES):
            j = _pi * len(_NN_PAULI_NAMES) + _pp
            B += rho_n[j] * la.expm(-1j * rho_n[j] * tau[j] * _PAIRS_2Q[_pn])
        dev = B.conj().T @ B - np.eye(4, dtype=complex)
        total += np.real(np.trace(dev.conj().T @ dev))
    return total

def apply_fake_backend_noise(params, layout):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]
    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), [t['qk_qa'], t['qk_qb']])
    qc_t = transpile(qc, _fake_backend, initial_layout=layout, optimization_level=1)
    qc_t.save_density_matrix(layout)
    result = _sim.run(qc_t, shots=1).result()
    return _qiskit_dm_to_numpy(np.array(result.data(0)['density_matrix']))

def noise_aware_objective(params, psi_target, layout):
    psi_ideal = compute_ideal_state(params)
    rho_noisy = apply_fake_backend_noise(params, layout)
    ideal_fid = float(np.abs(np.vdot(psi_target, psi_ideal)) ** 2
    noisy_fid = float(np.clip(np.real(psi_target.conj() @ rho_noisy @ psi_target), 0, 1))
    loss = 1.0 - noisy_fid + REG_LAMBDA * _upte_reg_penalty(params)
    return loss, noisy_fid, ideal_fid, rho_noisy

class MMAOptimizer:
    def __init__(self, n, move_limit=0.4, gamma=0.5):
        self.n, self.move_limit, self.gamma = n, move_limit, gamma
        self.L = self.U = self.prev_params = self.prev_loss = None
    def init(self, p0, delta=0.6):
        self.L, self.U = p0 - delta, p0 + delta
        self.prev_params = p0.copy()
    def step(self, x, grad):
        x_new = np.zeros_like(x)
        for i in range(self.n):
            g, xi, Li, Ui = grad[i], x[i], self.L[i], self.U[i]
            pi = abs(g) * (Ui - xi) ** 2 if g < 0 else 0.0
            qi = abs(g) * (xi - Li) ** 2 if g >= 0 else 0.0
            dn = pi / (Ui - xi + 1e-12) ** 2 + qi / (xi - Li + 1e-12) ** 2
            if dn > 1e-12:
                x_new[i] = xi + (pi / (Ui - xi + 1e-12) - qi / (xi - Li + 1e-12)) / dn
            else:
                x_new[i] = xi
            x_new[i] = np.clip(x_new[i], max(xi - self.move_limit, Li + 1e-6), min(xi + self.move_limit, Ui - 1e-6))
        return x_new
    def update(self, x, loss):
        good = self.prev_loss is None or loss < self.prev_loss - 1e-8
        s = 1.2 / self.gamma if good else self.gamma
        self.L = x - s * (x - self.L)
        self.U = x + s * (self.U - x)
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.prev_params = x.copy()
        self.prev_loss = loss

_prev_grad = None

def hybrid_gradient(params, psi_target, layout):
    global _prev_grad
    grad = np.zeros_like(params)
    for i in range(N_TERMS):
        pp = np.clip(params.copy(), 1e-6, None)
        pp[i] += 1e-4
        pm = np.clip(params.copy(), 1e-6, None)
        pm[i] -= 1e-4
        grad[i] = (noise_aware_objective(pp, psi_target, layout)[0] - noise_aware_objective(pm, psi_target, layout)[0]) / 2e-4
    for i in range(N_TERMS, N_PARAMS):
        pp = params.copy(); pp[i] += np.pi / 4
        pm = params.copy(); pm[i] -= np.pi / 4
        grad[i] = (noise_aware_objective(pp, psi_target, layout)[0] - noise_aware_objective(pm, psi_target, layout)[0]) / (np.pi / 2)
    if _prev_grad is None:
        _prev_grad = grad.copy()
        return grad
    g = 0.4 * grad + 0.6 * _prev_grad
    _prev_grad = g.copy()
    return g

def run_dual_mma(init_params, psi_target, layout, max_iter=MAX_ITERATIONS):
    global _prev_grad
    _prev_grad = None
    mma_r = MMAOptimizer(N_TERMS, move_limit=0.2); mma_r.init(init_params[:N_TERMS], delta=0.4)
    mma_t = MMAOptimizer(N_TERMS, move_limit=0.6); mma_t.init(init_params[N_TERMS:], delta=0.8)
    cur = init_params.copy()
    cur_loss, cur_fid, cur_ifid, _ = noise_aware_objective(cur, psi_target, layout)
    best_fid, best_params, stag = cur_fid, cur.copy(), 0
    for it in range(max_iter):
        grad = hybrid_gradient(cur, psi_target, layout)
        new = np.concatenate([
            mma_r.step(cur[:N_TERMS], grad[:N_TERMS]),
            mma_t.step(cur[N_TERMS:], grad[N_TERMS:])
        ])
        new_loss, new_fid, _, _ = noise_aware_objective(new, psi_target, layout)
        delta = new_fid - cur_fid
        if new_loss < cur_loss - 1e-6 or delta > -1e-5:
            cur, cur_loss, cur_fid = new, new_loss, new_fid
            if cur_fid > best_fid:
                best_fid, best_params, stag = cur_fid, cur.copy(), 0
            else:
                stag += 1
            mma_r.move_limit = min(0.4, mma_r.move_limit * (1.4 if delta > 0.01 else 1.2 if delta > 0.001 else 0.9))
        else:
            mma_r.move_limit = max(0.02, mma_r.move_limit * 0.8)
            stag += 1
        mma_t.move_limit = mma_r.move_limit * 2.0
        mma_r.update(cur[:N_TERMS], cur_loss)
        mma_t.update(cur[N_TERMS:], cur_loss)
        if (it + 1) % 5 == 0 or it == max_iter - 1:
            print(f"    Iter {it+1:2d}: NoisyFid={cur_fid:.4f} (Δ={delta:+.4f})")
        if stag > 15:
            print(f"    Optimisation Stagnated at iter {it+1}.")
            break
    print(f"  Optimisation Complete. Best NoisyFid: {best_fid:.4f}")
    return best_params

# ── Target state builder ───────────────────────────────────────────────────────
def build_target(seed=None):
    rng = np.random.default_rng(seed)
    terms = []
    for i in range(N_QUBITS - 1):
        for p in ['XX', 'YY', 'ZZ']:
            s = ['I'] * N_QUBITS
            s[i] = p[0]
            s[i + 1] = p[1]
            terms.append(''.join(s))
    return la.expm(-1j * SparsePauliOp(terms, coeffs=rng.uniform(-0.5, 0.5, len(terms))).to_matrix(sparse=False))

# ── Circuit builder ────────────────────────────────────────────────────────
def build_circuit(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]
    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), [t['qk_qa'], t['qk_qb']])
    return qc

# ── Fidelity evaluator (ZNE support) ───────────────────────────────────────
paulis = []
coeffs = []
for bits in itertools.product([0, 1], repeat=N_QUBITS):
    p = ''.join(['Z' if b else 'I' for b in bits])
    paulis.append(p)
    coeffs.append(1.0 / (2 ** N_QUBITS))
projector_obs = SparsePauliOp(paulis, coeffs=coeffs)

def eval_fidelity(qc_ansatz, layout, use_zne=False):
    qc_eval = qc_ansatz.copy()
    qc_eval.append(UnitaryGate(U_target.conj().T), range(N_QUBITS))
    qc_t = transpile(qc_eval, _fake_backend, initial_layout=layout, optimization_level=1)
    res_level = 2 if use_zne else 0
    estimator = EstimatorV2(_fake_backend, options={'resilience_level': res_level})
    try:
        job = estimator.run([(qc_t, projector_obs)])
        result = job.result()[0]
        return float(result.data.evs)
    except Exception as e:
        print(f"Estimator error: {e}")
        return 0.0

# ── Main execution ───────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print(f"RUNNING SINGLE SEED ({RUN_SEED}) BENCHMARK ON NOISIEST 8 LAYOUT")
    print("=" * 70)

    # Build target state
    U_target = build_target(seed=0)
    psi_0 = np.zeros(DIM, dtype=complex); psi_0[0] = 1.0
    psi_target = U_target @ psi_0

    # Initialise parameters
    rng = np.random.default_rng(RUN_SEED + 42)
    init_params = np.concatenate([
        rng.uniform(0.1, 0.4, N_TERMS),
        rng.uniform(0.1, np.pi, N_TERMS)
    ])

    # Baseline circuit (used only for TOPAZ optimisation)
    qc_baseline = build_circuit(init_params)

    # TOPAZ optimisation
    print("\n[Step 1/2] Running TOPAZ Structural MMA Optimisation (Seed {RUN_SEED})...")
    t0 = time.time()
    best_params = run_dual_mma(init_params, psi_target, noisiest_8, MAX_ITERATIONS)
    print(f"TOPAZ MMA optimisation completed in {time.time() - t0:.1f}s.")
    qc_topaz = build_circuit(best_params)

    # ----- ONLY ZNE AND ZNE+TOPAZ -----
    print("\n[Step 2/2] Running ZNE‑only and ZNE+TOPAZ evaluations...\n")

    # ZNE alone (using the baseline circuit)
    fid_zne_alone = eval_fidelity(qc_baseline, noisiest_8, use_zne=True)
    print(f"ZNE alone fidelity: {fid_zne_alone:.4f}")

    # TOPAZ + ZNE
    fid_topaz_zne = eval_fidelity(qc_topaz, noisiest_8, use_zne=True)
    print(f"TOPAZ + ZNE fidelity: {fid_topaz_zne:.4f}")

    print("\n" + "=" * 70)
    print("  Summary of ZNE evaluations")
    print("-" * 70)
    print(f"  ZNE alone               : {fid_zne_alone:.4f}")
    print(f"  TOPAZ + ZNE (optimised) : {fid_topaz_zne:.4f}")
    print("=" * 70)


SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (213582469.py, line 109)